# Import

In [1]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)

# Download data

In [11]:
noncurated_path = "../non_curated/h5ad/arce_2025.h5ad"
noncurated_dir = "../supplementary/arce_2025/"
download_file(
    url="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278572/suppl/GSE278572%5Fbarcodes.tsv.gz",
    dest_path=noncurated_dir+"GSE278572_barcodes.tsv.gz"
)
download_file(
    url="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278572/suppl/GSE278572%5Ffeatures.tsv.gz",
    dest_path=noncurated_dir+"GSE278572_features.tsv.gz"
)
download_file(
    url="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE278nnn/GSE278572/suppl/GSE278572%5Fmatrix.mtx.gz",
    dest_path=noncurated_dir+"GSE278572_matrix.mtx.gz"
)
download_file(
    url="https://zenodo.org/records/13924126/files/data_tables.zip?download=1",
    dest_path=noncurated_dir+"GSE278572_supplementary_tables.zip",
    unarchive=True
)
download_file(
    url="https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE278572&format=file&file=GSE278572%5Fprotospacer%5Fcalls%5Fper%5Fcell%2Ecsv%2Egz",
    dest_path=noncurated_dir+"GSE278572_protospacer_calls_per_cell.csv.gz",
    unarchive=False
)
!mv ../supplementary/arce_2025/data_tables ../supplementary/arce_2025/supplementary_tables

File ../supplementary/arce_2025/GSE278572_barcodes.tsv.gz already exists. Skipping download.
File ../supplementary/arce_2025/GSE278572_features.tsv.gz already exists. Skipping download.
File ../supplementary/arce_2025/GSE278572_matrix.mtx.gz already exists. Skipping download.
File ../supplementary/arce_2025/GSE278572_supplementary_tables.zip already exists. Skipping download.
File ../supplementary/arce_2025/GSE278572_protospacer_calls_per_cell.csv.gz already exists. Skipping download.
mv: cannot stat '../supplementary/arce_2025/data_tables': No such file or directory


# Convert to h5ad

Uncomment if running for the first time

In [ ]:
# import scanpy as sc
# adata = sc.read_10x_mtx(
#     noncurated_dir,
#     var_names='gene_symbols',
#     make_unique=False,
#     gex_only=True,
#     prefix="GSE278572_"
# )

# # write to h5ad
# adata.write_h5ad(noncurated_path)

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
... storing 'feature_types' as categorical


# Initialise the dataset object

In [4]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from ../non_curated/h5ad/arce_2025.h5ad


/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


### Add cell barcodes to the obs slot

In [6]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str)

print(cur_data.adata.obs[['cell_barcode']].head())

                          cell_barcode
AAACCTGAGAACAACT-1  AAACCTGAGAACAACT-1
AAACCTGAGAAGAAGC-1  AAACCTGAGAAGAAGC-1
AAACCTGAGAGAACAG-1  AAACCTGAGAGAACAG-1
AAACCTGAGAGAGCTC-1  AAACCTGAGAGAGCTC-1
AAACCTGAGAGCTATA-1  AAACCTGAGAGCTATA-1


# OBS slot curation

### Show unique perturbations

In [6]:
# cur_data.rename_columns(slot = 'obs', name_dict = {'sgID_AB': 'perturbation_name'})

### Add guide RNA information

Orignal guideRNA metadata referenced in the paper cannot be unambiguously mapped. Requested updated metadata from the authors - it is contained within the `data_exploration/Perturbseq/supplementary/arce_2025/feature_reference_name_key.csv` folder.

In [8]:
supp_guides = pd.read_csv("../supplementary/arce_2025/feature_reference_name_key.csv")
supp_guides["guide_sequence"] = supp_guides["sgRNA_long"].str.split("_").str[1]
supp_guides = supp_guides[
    ["id", "target_gene_id", "target_gene_name", "guide_sequence"]
]
supp_guides = supp_guides.replace(
    {"target_gene_id": {"Non-Targeting": "control_nontargeting"},
     "target_gene_name": {"Non-Targeting": "control_nontargeting"}}
)
supp_guides

,id,target_gene_id,target_gene_name,guide_sequence
0,BACH2_1_CRISPRi,ENSG00000112182,BACH2,GCGCTGTGCGACCGCAGCCC
1,BACH2_2_CRISPRi,ENSG00000112182,BACH2,CAGCAGCGGCCGTGCACGCC
2,BATF_1_CRISPRi,ENSG00000156127,BATF,CCTGCGTCCTCCTCACTCTG
3,BATF_2_CRISPRi,ENSG00000156127,BATF,TCCCTCTGCACCCCAGAGTG
4,BPTF_1_CRISPRi,ENSG00000171634,BPTF,AAGGCTCAATCCGAATTGCT
...,...,...,...,...
60,Non-Targeting_5_CRISPRi,control_nontargeting,control_nontargeting,ACGCCTCCTCAAATTAGCTC
61,Non-Targeting_6_CRISPRi,control_nontargeting,control_nontargeting,GAAACGAGAAGTTTGTACTA
62,Non-Targeting_7_CRISPRi,control_nontargeting,control_nontargeting,TAATGCTGCACACGCCGAAT
63,Non-Targeting_8_CRISPRi,control_nontargeting,control_nontargeting,GGCTGGTTGACGACTCCTGA


In [12]:
# read in the guide RNA spreadsheet
# guides for the essential library are in "ST20" sheet
guide_info_df = pd.read_csv(
    "../supplementary/arce_2025/GSE278572_protospacer_calls_per_cell.csv.gz"
)
guide_info_df = guide_info_df.rename(columns={"feature_call": "perturbation_name"})
guide_info_df = (
    guide_info_df[["cell_barcode", "perturbation_name"]]
    .drop_duplicates()
    .assign(perturbation_name=guide_info_df["perturbation_name"].str.split("|"))
    .explode("perturbation_name")
    .merge(supp_guides, how="left", left_on="perturbation_name", right_on="id")
    .drop(columns=["id"])
    .groupby("cell_barcode", as_index=False)
    .agg({
        "perturbation_name": lambda x: "|".join(x),
        "target_gene_name": lambda x: "|".join(x),
        "target_gene_id": lambda x: "|".join(x),
        "guide_sequence": lambda x: "|".join(x)
    })
)
guide_info_df

,cell_barcode,perturbation_name,target_gene_name,target_gene_id,guide_sequence
0,AAACCTGAGAAACCTA-2,IRF1_2_CRISPRi,IRF1,ENSG00000125347,CGGCCGGCGTGGACTGGGCA
1,AAACCTGAGAAAGTGG-4,MED11_2_CRISPRi|USP22_1_CRISPRi,MED11|USP22,ENSG00000161920|ENSG00000124422,GTAGGTAGCCATTATCACTC|GCGCCGAGAACAAAGCGCGG
2,AAACCTGAGAACAACT-1,IRF1_1_CRISPRi|MED11_1_CRISPRi|MED12_2_CRISPRi,IRF1|MED11|MED12,ENSG00000125347|ENSG00000161920|ENSG00000184634,GCCCGAGCCCCGCCGAACCG|GAACAAGCGTCGCGTTTCTG|GGCG...
3,AAACCTGAGAACAACT-8,KLF13_1_CRISPRi,KLF13,ENSG00000169926,CCGGTTCTAAGGATGCCGAG
4,AAACCTGAGAACTCGG-3,BPTF_2_CRISPRi|IRF4_2_CRISPRi|MEF2D_1_CRISPRi|...,BPTF|IRF4|MEF2D|KLF2,ENSG00000171634|ENSG00000137265|ENSG0000011660...,GATGGCGGCTGAAGGCGATC|TCGGAGCTGAGGGCAGCGGT|GCCG...
...,...,...,...,...,...
191615,TTTGTCATCTTAGAGC-4,Non-Targeting_4_CRISPRi,control_nontargeting,control_nontargeting,GCTGTTCCGAAGTTGAGAAT
191616,TTTGTCATCTTGAGAC-5,BACH2_1_CRISPRi|CBFB_1_CRISPRi|NFKB2_2_CRISPRi,BACH2|CBFB|NFKB2,ENSG00000112182|ENSG00000067955|ENSG00000077150,GCGCTGTGCGACCGCAGCCC|GCGGCAGGCAACGGCTGAGG|CGGA...
191617,TTTGTCATCTTGAGGT-8,BATF_2_CRISPRi|ATXN7L3_1_CRISPRi,BATF|ATXN7L3,ENSG00000156127|ENSG00000087152,TCCCTCTGCACCCCAGAGTG|ACTGCTCGCGCCTGCTAGAA
191618,TTTGTCATCTTGTATC-1,CBFB_2_CRISPRi,CBFB,ENSG00000067955,GGCAACGGCTGAGGCGGCGG


In [13]:
cur_data.adata.obs = cur_data.adata.obs.merge(
    guide_info_df,
    how='left',
    left_on='cell_barcode',
    right_on='cell_barcode'
)
cur_data.adata.obs

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,cell_barcode,perturbation_name,target_gene_name,target_gene_id,guide_sequence
0,AAACCTGAGAACAACT-1,IRF1_1_CRISPRi|MED11_1_CRISPRi|MED12_2_CRISPRi,IRF1|MED11|MED12,ENSG00000125347|ENSG00000161920|ENSG00000184634,GCCCGAGCCCCGCCGAACCG|GAACAAGCGTCGCGTTTCTG|GGCG...
1,AAACCTGAGAAGAAGC-1,CBFB_2_CRISPRi|Non-Targeting_6_CRISPRi,CBFB|control_nontargeting,ENSG00000067955|control_nontargeting,GGCAACGGCTGAGGCGGCGG|GAAACGAGAAGTTTGTACTA
2,AAACCTGAGAGAACAG-1,NaN,NaN,NaN,NaN
3,AAACCTGAGAGAGCTC-1,NaN,NaN,NaN,NaN
4,AAACCTGAGAGCTATA-1,IRF4_2_CRISPRi|KLF13_1_CRISPRi|MYB_1_CRISPRi,IRF4|KLF13|MYB,ENSG00000137265|ENSG00000169926|ENSG00000118513,TCGGAGCTGAGGGCAGCGGT|CCGGTTCTAAGGATGCCGAG|GGAG...
...,...,...,...,...,...
249794,TTTGTCATCGGATGGA-8,PRDM1_2_CRISPRi|TAF5L_2_CRISPRi|Non-Targeting_...,PRDM1|TAF5L|control_nontargeting,ENSG00000057657|ENSG00000135801|control_nontar...,GGCCCTCCAGTGTTGCGGAG|CGGCCGCCCAGAGCGGCGGC|TAAT...
249795,TTTGTCATCTCAAGTG-8,MED12_1_CRISPRi,MED12,ENSG00000184634,ACGGCGGCCGAGAGACAACA
249796,TTTGTCATCTCCGGTT-8,KLF13_1_CRISPRi,KLF13,ENSG00000169926,CCGGTTCTAAGGATGCCGAG
249797,TTTGTCATCTTGAGGT-8,BATF_2_CRISPRi|ATXN7L3_1_CRISPRi,BATF|ATXN7L3,ENSG00000156127|ENSG00000087152,TCCCTCTGCACCCCAGAGTG|ACTGCTCGCGCCTGCTAGAA


### Replace NaNs with control_casonly

In [14]:
cur_data.adata.obs[cur_data.adata.obs.isna()] = 'control_casonly'
cur_data.adata.obs

,cell_barcode,perturbation_name,target_gene_name,target_gene_id,guide_sequence
0,AAACCTGAGAACAACT-1,IRF1_1_CRISPRi|MED11_1_CRISPRi|MED12_2_CRISPRi,IRF1|MED11|MED12,ENSG00000125347|ENSG00000161920|ENSG00000184634,GCCCGAGCCCCGCCGAACCG|GAACAAGCGTCGCGTTTCTG|GGCG...
1,AAACCTGAGAAGAAGC-1,CBFB_2_CRISPRi|Non-Targeting_6_CRISPRi,CBFB|control_nontargeting,ENSG00000067955|control_nontargeting,GGCAACGGCTGAGGCGGCGG|GAAACGAGAAGTTTGTACTA
2,AAACCTGAGAGAACAG-1,control_casonly,control_casonly,control_casonly,control_casonly
3,AAACCTGAGAGAGCTC-1,control_casonly,control_casonly,control_casonly,control_casonly
4,AAACCTGAGAGCTATA-1,IRF4_2_CRISPRi|KLF13_1_CRISPRi|MYB_1_CRISPRi,IRF4|KLF13|MYB,ENSG00000137265|ENSG00000169926|ENSG00000118513,TCGGAGCTGAGGGCAGCGGT|CCGGTTCTAAGGATGCCGAG|GGAG...
...,...,...,...,...,...
249794,TTTGTCATCGGATGGA-8,PRDM1_2_CRISPRi|TAF5L_2_CRISPRi|Non-Targeting_...,PRDM1|TAF5L|control_nontargeting,ENSG00000057657|ENSG00000135801|control_nontar...,GGCCCTCCAGTGTTGCGGAG|CGGCCGCCCAGAGCGGCGGC|TAAT...
249795,TTTGTCATCTCAAGTG-8,MED12_1_CRISPRi,MED12,ENSG00000184634,ACGGCGGCCGAGAGACAACA
249796,TTTGTCATCTCCGGTT-8,KLF13_1_CRISPRi,KLF13,ENSG00000169926,CCGGTTCTAAGGATGCCGAG
249797,TTTGTCATCTTGAGGT-8,BATF_2_CRISPRi|ATXN7L3_1_CRISPRi,BATF|ATXN7L3,ENSG00000156127|ENSG00000087152,TCCCTCTGCACCCCAGAGTG|ACTGCTCGCGCCTGCTAGAA


### Standardise perturbation targets

In [15]:
cur_data.standardize_genes(
    slot='obs',
    input_column='target_gene_id',
    input_column_type='ensembl_gene_id',
    multiple_entries=True,
    remove_version=False,
    multiple_entries_sep='|'
    # version_sep='.'
)

--------------------------------------------------
Successfully mapped 30 out of 30 Ensembl IDs.
--------------------------------------------------
Collapsed column positional_index using separator |


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [16]:
cur_data.adata.obs.head()

,cell_barcode,target_gene_name,guide_sequence,target_gene_id,perturbation_name,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index
index,,,,,,,,,,,
0,AAACCTGAGAACAACT-1,IRF1|MED11|MED12,GCCCGAGCCCCGCCGAACCG|GAACAAGCGTCGCGTTTCTG|GGCG...,ENSG00000125347|ENSG00000161920|ENSG00000184634,IRF1_1_CRISPRi|MED11_1_CRISPRi|MED12_2_CRISPRi,ENSG00000125347|ENSG00000161920|ENSG00000184634,IRF1|MED11|MED12,protein_coding|protein_coding|protein_coding,chr5:132440440-132508719;-1|chr17:4731428-4733...,5|17|X,0|0|0
1,AAACCTGAGAAGAAGC-1,CBFB|control_nontargeting,GGCAACGGCTGAGGCGGCGG|GAAACGAGAAGTTTGTACTA,ENSG00000067955|control_nontargeting,CBFB_2_CRISPRi|Non-Targeting_6_CRISPRi,ENSG00000067955|control_nontargeting,CBFB|control_nontargeting,protein_coding|control_nontargeting,chr16:67028984-67101058;1|control_nontargeting,16|control_nontargeting,1|1
2,AAACCTGAGAGAACAG-1,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,2
3,AAACCTGAGAGAGCTC-1,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,control_casonly,3
4,AAACCTGAGAGCTATA-1,IRF4|KLF13|MYB,TCGGAGCTGAGGGCAGCGGT|CCGGTTCTAAGGATGCCGAG|GGAG...,ENSG00000137265|ENSG00000169926|ENSG00000118513,IRF4_2_CRISPRi|KLF13_1_CRISPRi|MYB_1_CRISPRi,ENSG00000137265|ENSG00000169926|ENSG00000118513,IRF4|KLF13|MYB,protein_coding|protein_coding|protein_coding,chr6:391739-411443;1|chr15:31326706-31435665;1...,6|15|6,4|4|4


### Add `perturbed_target_number` column

In [17]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

Counted entries in column perturbed_target_symbol of adata.obs and stored in perturbed_target_number


### Encode chromosomes as integers

In [18]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


In [19]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_chromosome_encoding'])

Observation data:
DataFrame shape: (249799, 2)
--------------------------------------------------
                                        perturbation_name  \
index                                                       
0          IRF1_1_CRISPRi|MED11_1_CRISPRi|MED12_2_CRISPRi   
1                  CBFB_2_CRISPRi|Non-Targeting_6_CRISPRi   
2                                         control_casonly   
3                                         control_casonly   
4            IRF4_2_CRISPRi|KLF13_1_CRISPRi|MYB_1_CRISPRi   
...                                                   ...   
249794  PRDM1_2_CRISPRi|TAF5L_2_CRISPRi|Non-Targeting_...   
249795                                    MED12_1_CRISPRi   
249796                                    KLF13_1_CRISPRi   
249797                   BATF_2_CRISPRi|ATXN7L3_1_CRISPRi   
249798                                    control_casonly   

        perturbed_target_chromosome_encoding  
index                                         
0             

### Add metadata information from supplementary table 14

In [20]:
supp_s14 = pd.read_excel(
    "../supplementary/arce_2025/supplementary_tables/S14_metadata_Treg_Teff_perturbseq.xlsx",
    sheet_name="metadata_Treg_total_Teff_total_",
)
supp_s14 = supp_s14[["cell", "donor", "HTO_classification"]]
supp_s14 = supp_s14.rename(
    columns={"cell": "cell_barcode", "donor": "biological_replicate"}
)
supp_s14

,cell_barcode,biological_replicate,HTO_classification
0,AAACCTGAGTCTCCTC-1,B,Resting-Teff
1,AAACCTGCAACCGCCA-1,B,Resting-Teff
2,AAACCTGCACCCAGTG-1,A,Resting-Teff
3,AAACCTGGTCCAAGTT-1,A,Resting-Teff
4,AAACCTGGTGCACGAA-1,A,Resting-Teff
...,...,...,...
100082,TTTGTCAGTCCTAGCG-8,A,Stimulated-Treg
100083,TTTGTCAGTGATAAAC-8,A,Stimulated-Treg
100084,TTTGTCAGTGCCTGCA-8,A,Stimulated-Treg
100085,TTTGTCAGTTCGTTGA-8,A,Stimulated-Treg


In [21]:
cur_data.adata.obs = cur_data.adata.obs.merge(supp_s14, how="left", on="cell_barcode")

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Drop unmatched cells - these don't have metadata

In [22]:
cur_data.adata = cur_data.adata[cur_data.adata.obs['biological_replicate'].notna(),:]

In [23]:
cur_data.adata.obs

,cell_barcode,target_gene_name,guide_sequence,target_gene_id,perturbation_name,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding,biological_replicate,HTO_classification
14,AAACCTGAGTCTCCTC-1,DNMT1,GGGCAGCGAGATGGCCGGGA,ENSG00000130816,DNMT1_1_CRISPRi,ENSG00000130816,DNMT1,protein_coding,chr19:10133342-10231286;-1,19,14,1,19,B,Resting-Teff
15,AAACCTGCAAAGGAAG-1,USP22,GCGCCGAGAACAAAGCGCGG,ENSG00000124422,USP22_1_CRISPRi,ENSG00000124422,USP22,protein_coding,chr17:20999596-21043760;-1,17,15,1,17,A,Stimulated-Treg
16,AAACCTGCAACCGCCA-1,MYC,AGGCAGAGGGAGCGAGCGGG,ENSG00000136997,MYC_1_CRISPRi,ENSG00000136997,MYC,protein_coding,chr8:127735434-127742951;1,8,16,1,8,B,Resting-Teff
19,AAACCTGCAACTGGCC-1,STAT5B,CCAGCGCAGGCAACTCCGCG,ENSG00000173757,STAT5B_2_CRISPRi,ENSG00000173757,STAT5B,protein_coding,chr17:42199176-42288633;-1,17,19,1,17,B,Stimulated-Treg
21,AAACCTGCAAGCCCAC-1,PRDM1,GGCCCTCCAGTGTTGCGGAG,ENSG00000057657,PRDM1_2_CRISPRi,ENSG00000057657,PRDM1,protein_coding,chr6:105993463-106109939;1,6,21,1,6,B,Resting-Treg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249785,TTTGTCATCAGTTAGC-8,USP22,GCGCCGAGAACAAAGCGCGG,ENSG00000124422,USP22_1_CRISPRi,ENSG00000124422,USP22,protein_coding,chr17:20999596-21043760;-1,17,249785,1,17,B,Stimulated-Teff
249787,TTTGTCATCATGTGGT-8,ATXN7L3,GAGCGCGTGCATCTGCCCCG,ENSG00000087152,ATXN7L3_2_CRISPRi,ENSG00000087152,ATXN7L3,protein_coding,chr17:44191805-44200961;-1,17,249787,1,17,A,Resting-Teff
249788,TTTGTCATCCACGAAT-8,PRDM1,AGAGGCAAGAGCAGCGACCG,ENSG00000057657,PRDM1_1_CRISPRi,ENSG00000057657,PRDM1,protein_coding,chr6:105993463-106109939;1,6,249788,1,6,B,Resting-Treg
249795,TTTGTCATCTCAAGTG-8,MED12,ACGGCGGCCGAGAGACAACA,ENSG00000184634,MED12_1_CRISPRi,ENSG00000184634,MED12,protein_coding,chrX:71118543-71144103;1,X,249795,1,23,B,Resting-Teff


### Add metadata

In [24]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        "dataset_id": cur_data.dataset_id,
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        # perturbation type
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        "data_modality": "Perturb-seq",
        "significant": None,
        "significance_criteria": None,
        "score_interpretation": None,

        # treatment
        # "treatment_label": None,
        # "treatment_id": None,
        # replicates
        "technical_replicate": None,
        # "biological_replicate": None,
        # model system
        "model_system_label": "primary_cell",
        "model_system_id": None,
        "tissue": "lymphoid tissue",
        "cell_line_label": None,
        "cell_line_id": None,
        # "cell_type_label": None,
        "disease_label": "healthy",
        "disease_id": None,

        "timepoint": "P10DT0H0M0S",
        "species": "Homo sapiens",
        "sex_label": None,
        "sex_id": None,
        "developmental_stage_label": None,
        "developmental_stage_id": None,

        "study_title": "Central control of dynamic gene circuits governs T cell rest and activation",
        "study_uri": "https://doi.org/10.1038/s41586-024-08314-y",
        "study_year": 2025,
        "first_author": "Maya M. Arce",
        "last_author": "Alexander Marson",

        "experiment_title": "Perturb-seq of primary human CD4+ T regulatory and T effector cells under resting and stimulated conditions",
        "experiment_summary": """
            Isolated human Tregs and Teffs from two healthy donors were stimulated with ImmunoCult CD3/CD28/CD2 activator and sequentially transduced with dCas9-KRAB-Zim3 lentivirus (24 hours post-stimulation) and a Perturb-seq guide library (48 hours post-stimulation, MOI 0.3). The library consisted of 28 regulators of IL-2Rα which were subset from Dolcetto library. Cells underwent continuous blasticidin selection starting 48 hours after the first transduction. On day 8, half of the cultures were restimulated using ImmunoCult CD3/CD28/CD2 activator. On day 10, cells were pooled by condition, sorted for live GFP+ expression, and stained with a TotalSeq™-C Universal Cocktail containing hashtag antibodies to distinguish cell types (CD4+ T-effector and T-regulatory) and stimulation conditions (Resting vs. Stimulated). The samples were subsequently processed using the 10x Genomics Chromium Next GEM Single Cell 5' HT v2 platform with Feature Barcode technology and sequenced on an Illumina NovaSeqX.
        """,

        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],

        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",

        "library_generation_method_id": None,
        "library_generation_method_label": "dCas9-KRAB-Zim3",

        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",

        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",

        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",

        "library_name": "Human CRISPR Inhibition Pooled Library (Dolcetto)",
        "library_uri": "https://www.addgene.org/pooled-library/broadgpp-human-crispri-dolcetto/",

        "library_format_id": None,
        "library_format_label": "pooled",

        "library_scope_id": None,
        "library_scope_label": "focused",

        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",

        "library_manufacturer": "Doench lab",
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "2",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()),
        "library_total_variants": None,

        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",

        "readout_type_id": None,
        "readout_type_label": "transcriptomic",

        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",

        "method_name_id": None,
        "method_name_label": "Perturb-seq",

        "method_uri": None,

        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Chromium Next GEM Single Cell 5-prime HT Kit v2",

        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina NovaSeq X",

        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",

        "software_counts_id": None,
        "software_counts_label": "CellRanger",

        "software_analysis_id": None,
        "software_analysis_label": "Seurat",

        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        
        "license_label": "free to use license",
        "license_id": "SWO:1000061",

        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE278572",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE278572",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE278572_*.*",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column technical_replicate added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column tissue added to adata.obs
Column cell_line_label added to adata.obs
Column cell_line_id added to adata.obs
Column disease_label added to adata.obs
Column disease_id added to adata.obs
Column timepoint added to adata.obs
Column species added to adata.obs
Column sex_label added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_label added to adata.obs
Column developmental_stage_id added to adata.obs
Column study_title added to adata.obs
Column study_uri added to adata.obs
Column study_year a

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


### Curate tissue information


In [25]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 1 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
      input_column input_column_lower       name_lower     ontology_id
0  lymphoid tissue    lymphoid tissue  lymphoid tissue  UBERON:0001744
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell type information

In [26]:
cur_data.adata.obs["cell_type"] = cur_data.adata.obs["HTO_classification"].replace(
    {
        "Resting-Teff": "effector CD4-positive, alpha-beta T cell",
        "Stimulated-Teff": "effector CD4-positive, alpha-beta T cell",
        "Resting-Treg": "CD4-positive, CD25-positive, alpha-beta regulatory T cell",
        "Stimulated-Treg": "CD4-positive, CD25-positive, alpha-beta regulatory T cell"
    }
)


In [27]:
cur_data.standardize_ontology(
    input_column='cell_type',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

Mapped 2 cell_type ontology terms from `cell_type` column to ontology terms
DataFrame shape: (2, 4)
--------------------------------------------------
                                        input_column  \
0           effector CD4-positive, alpha-beta T cell   
1  CD4-positive, CD25-positive, alpha-beta regula...   

                                  input_column_lower  \
0           effector cd4-positive, alpha-beta t cell   
1  cd4-positive, cd25-positive, alpha-beta regula...   

                                          name_lower ontology_id  
0           effector cd4-positive, alpha-beta t cell  CL:0001044  
1  cd4-positive, cd25-positive, alpha-beta regula...  CL:0000792  
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell line information

In [28]:
# cur_data.standardize_ontology(
#     input_column='cell_line_label',
#     column_type='term_name',
#     ontology_type='cell_line',
#     overwrite=True
# )

### Curate disease information

In [29]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Curate treatment information

In [30]:
cur_data.adata.obs["treatment_label"] = cur_data.adata.obs["HTO_classification"].replace(
    {
        "Resting-Teff": "Untreated Control",
        "Resting-Treg": "Untreated Control",
        "Stimulated-Teff": "anti-CD3 antibody|anti-CD28 antibody|anti-CD2 antibody",
        "Stimulated-Treg": "anti-CD3 antibody|anti-CD28 antibody|anti-CD2 antibody",
    }
)
cur_data.adata.obs["treatment_id"] = cur_data.adata.obs["HTO_classification"].replace(
    {
        "Resting-Teff": "NCIT:C184729",
        "Resting-Treg": "NCIT:C184729",
        "Stimulated-Teff": "EFO:0003317|EFO:0003304|NCIT:C184729",
        "Stimulated-Treg": "EFO:0003317|EFO:0003304|NCIT:C184729",
    }
)

### Match schema column order

In [31]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [32]:
cur_data.validate_data(slot='obs', verbose=True)

2026-04-28 13:29:51,226 INFO curation_tools.curation_tools: adata.obs is valid according to the obs_schema.


,dataset_id,sample_id,cell_barcode,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,...,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,arce_2025,1,AAACCTGAGTCTCCTC-1,Perturb-seq,<NA>,<NA>,DNMT1_1_CRISPRi,chr19:10133342-10231286;-1,19,19,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
1,arce_2025,2,AAACCTGCAAAGGAAG-1,Perturb-seq,<NA>,<NA>,USP22_1_CRISPRi,chr17:20999596-21043760;-1,17,17,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
2,arce_2025,3,AAACCTGCAACCGCCA-1,Perturb-seq,<NA>,<NA>,MYC_1_CRISPRi,chr8:127735434-127742951;1,8,8,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
3,arce_2025,4,AAACCTGCAACTGGCC-1,Perturb-seq,<NA>,<NA>,STAT5B_2_CRISPRi,chr17:42199176-42288633;-1,17,17,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
4,arce_2025,5,AAACCTGCAAGCCCAC-1,Perturb-seq,<NA>,<NA>,PRDM1_2_CRISPRi,chr6:105993463-106109939;1,6,6,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100082,arce_2025,100083,TTTGTCATCAGTTAGC-8,Perturb-seq,<NA>,<NA>,USP22_1_CRISPRi,chr17:20999596-21043760;-1,17,17,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
100083,arce_2025,100084,TTTGTCATCATGTGGT-8,Perturb-seq,<NA>,<NA>,ATXN7L3_2_CRISPRi,chr17:44191805-44200961;-1,17,17,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
100084,arce_2025,100085,TTTGTCATCCACGAAT-8,Perturb-seq,<NA>,<NA>,PRDM1_1_CRISPRi,chr6:105993463-106109939;1,6,6,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061
100085,arce_2025,100086,TTTGTCATCTCAAGTG-8,Perturb-seq,<NA>,<NA>,MED12_1_CRISPRi,chrX:71118543-71144103;1,X,23,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE278572"", ""dataset_u...",free to use license,SWO:1000061


# VAR slot curation

### Standardise genes

In [33]:
cur_data.show_var()

Variable data:
DataFrame shape: (36601, 2)
--------------------------------------------------
                    gene_ids    feature_types
MIR1302-2HG  ENSG00000243485  Gene Expression
FAM138A      ENSG00000237613  Gene Expression
OR4F5        ENSG00000186092  Gene Expression
AL627309.1   ENSG00000238009  Gene Expression
AL627309.3   ENSG00000239945  Gene Expression
...                      ...              ...
AC141272.1   ENSG00000277836  Gene Expression
AC023491.2   ENSG00000278633  Gene Expression
AC007325.1   ENSG00000276017  Gene Expression
AC007325.4   ENSG00000278817  Gene Expression
AC007325.2   ENSG00000277196  Gene Expression

[36601 rows x 2 columns]
--------------------------------------------------


In [34]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ids",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

Missing Ensembl IDs: ['ENSG00000273203', 'ENSG00000272979', 'ENSG00000267243', 'ENSG00000261470', 'ENSG00000237954', 'ENSG00000258170', 'ENSG00000243179', 'ENSG00000278878', 'ENSG00000228334', 'ENSG00000275249', 'ENSG00000263316', 'ENSG00000273301', 'ENSG00000229628', 'ENSG00000231255', 'ENSG00000235450', 'ENSG00000286997', 'ENSG00000251679', 'ENSG00000237975', 'ENSG00000229694', 'ENSG00000287906', 'ENSG00000182584', 'ENSG00000264078', 'ENSG00000250431', 'ENSG00000251518', 'ENSG00000286895', 'ENSG00000272592', 'ENSG00000228216', 'ENSG00000229964', 'ENSG00000249069', 'ENSG00000224228', 'ENSG00000215159', 'ENSG00000288436', 'ENSG00000235665', 'ENSG00000274307', 'ENSG00000248359', 'ENSG00000224247', 'ENSG00000286792', 'ENSG00000249125', 'ENSG00000229160', 'ENSG00000272146', 'ENSG00000230699', 'ENSG00000267637', 'ENSG00000270178', 'ENSG00000232597', 'ENSG00000277010', 'ENSG00000259436', 'ENSG00000236451', 'ENSG00000258196', 'ENSG00000236453', 'ENSG00000286564', 'ENSG00000241084', 'ENSG0000

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [35]:
cur_data.show_var()


Variable data:
DataFrame shape: (36601, 5)
--------------------------------------------------
         feature_types         gene_ids  ensembl_gene_id  gene_symbol  \
index                                                                   
0      Gene Expression  ENSG00000243485  ENSG00000243485  MIR1302-2HG   
1      Gene Expression  ENSG00000237613  ENSG00000237613      FAM138A   
2      Gene Expression  ENSG00000186092  ENSG00000186092        OR4F5   
3      Gene Expression  ENSG00000238009  ENSG00000241860          NaN   
4      Gene Expression  ENSG00000239945  ENSG00000239945         None   
...                ...              ...              ...          ...   
36596  Gene Expression  ENSG00000277836  ENSG00000277836          NaN   
36597  Gene Expression  ENSG00000278633  ENSG00000278633          NaN   
36598  Gene Expression  ENSG00000276017  ENSG00000276017          NaN   
36599  Gene Expression  ENSG00000278817  ENSG00000278817          NaN   
36600  Gene Expression  ENSG00

### Replace unmapped gene symbols with original gene symbols

In [37]:
cur_data.adata.var['gene_symbol'] = cur_data.adata.var['gene_symbol'].fillna(
    cur_data.adata.var['original_index']
)

In [38]:
cur_data.adata.var

,feature_types,gene_ids,ensembl_gene_id,gene_symbol,original_index
index,,,,,
0,Gene Expression,ENSG00000243485,ENSG00000243485,MIR1302-2HG,MIR1302-2HG
1,Gene Expression,ENSG00000237613,ENSG00000237613,FAM138A,FAM138A
2,Gene Expression,ENSG00000186092,ENSG00000186092,OR4F5,OR4F5
3,Gene Expression,ENSG00000238009,ENSG00000241860,AL627309.1,AL627309.1
4,Gene Expression,ENSG00000239945,ENSG00000239945,AL627309.3,AL627309.3
...,...,...,...,...,...
36596,Gene Expression,ENSG00000277836,ENSG00000277836,AC141272.1,AC141272.1
36597,Gene Expression,ENSG00000278633,ENSG00000278633,AC023491.2,AC023491.2
36598,Gene Expression,ENSG00000276017,ENSG00000276017,AC007325.1,AC007325.1


### Validate var metadata

In [39]:
cur_data.validate_data(slot='var')

2026-04-28 13:40:38,754 INFO curation_tools.curation_tools: adata.var is valid according to the var_schema.


,ensembl_gene_id,gene_symbol
index,,
0,ENSG00000243485,MIR1302-2HG
1,ENSG00000237613,FAM138A
2,ENSG00000186092,OR4F5
3,ENSG00000241860,AL627309.1
4,ENSG00000239945,AL627309.3
...,...,...
36596,ENSG00000277836,AC141272.1
36597,ENSG00000278633,AC023491.2
36598,ENSG00000276017,AC007325.1


# Save the dataset

In [40]:
cur_data.save_curated_data_h5ad()

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)
... storing 'dataset_id' as categorical
... storing 'data_modality' as categorical
... storing 'significance_criteria' as categorical
... storing 'perturbation_name' as categorical
... storing 'perturbed_target_coord' as categorical
... storing 'perturbed_target_chromosome' as categorical
... storing 'perturbed_target_ensg' as categorical
... storing 'perturbed_target_symbol' as categorical
... storing 'perturbed_target_biotype' as categorical
... storing 'guide_sequence' as categorical
... storing 'perturbation_type_label' as categorical
... storing 'perturbation

✅ Curated h5ad data saved to ../curated/h5ad/arce_2025_curated.h5ad


In [41]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to ../curated/parquet/arce_2025_curated_metadata.parquet


# Upload to BigQuery

In [42]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/arce_2025_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

Staging table: loading `.parquet` file ../curated/parquet/arce_2025_curated_metadata.parquet to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging...
Staging table: loaded 100087 rows to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.perturb_seq.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging


# Upload to GC Storage

In [43]:
!gcloud storage cp ../curated/h5ad/arce_2025_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../curated/h5ad/arce_2025_curated.h5ad to gs://perturbation-catalogue-lake/perturbseq/curated/arce_2025_curated.h5ad
  Completed files 32/1 | 3.3GiB/3.3GiB | 313.9MiB/s                            

Average throughput: 328.3MiB/s
